# Sumativa 3 — Taller II

## Integración de fuentes heterogéneas con Apache Spark

**Curso:** Gestión de Datos y Tecnologías Big Data (MCDI502)  
**Grupo:** 7  
**Archivo:** `mcdi502_s3_grupo7.ipynb`

Este notebook sigue la rúbrica de la Sumativa 3 y adapta los ejemplos entregados por el docente. Cada sección indica qué fuente se conecta, qué variables incorpora y qué resultado debe mostrar.

## Ruta de trabajo

1. Preparar el entorno y crear SparkSession/SparkContext.
2. Ingerir 100 observaciones de cada fuente, cuando la fuente las tenga disponibles.
3. Transformar las dos columnas requeridas.
4. Integrar las cuatro fuentes con una clave posicional documentada.
5. Crear vistas temporales y ejecutar las consultas C1–C7.

**Criterio de datos:** el notebook no completa registros faltantes mediante duplicación o datos inventados. Si una fuente entrega menos de 100 filas, lo informa para que se solicite al docente una fuente con cobertura suficiente.

## Fuentes y variables

| Fuente | DataFrame | Variables incluidas |
|---|---|---|
| OpenMeteo | `df_openmeteo` | Temperatura, humedad, precipitación, viento, dirección e índice UV |
| AQICN | `df_aqicn` | PM2.5, PM10, O3, NO2 y SO2 |
| MongoDB (ESP32) | `df_mongo_bpm` | Altura en pies, presión atmosférica, monóxido de carbono y fecha |
| MySQL (ESP32 vía API REST) | `df_mysql_dht` | Temperatura interior, humedad, identificador y fecha |

Los CSV de ventas incorporados como material adicional no se usan aquí porque no son parte de las cuatro fuentes exigidas por esta rúbrica.

## 1. Configuración del entorno

Se utiliza Google Colab y PySpark 3.5.7, siguiendo el entorno del notebook docente de MongoDB. La instalación incluye Java, Spark y los conectores MongoDB.

In [ ]:
!wget -q https://downloads.apache.org/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar xf spark-3.5.7-bin-hadoop3.tgz
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark==3.5.7 findspark requests

# Conectores usados en el ejemplo docente de MongoDB
!wget -q https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/3.0.1/mongo-spark-connector_2.12-3.0.1.jar -P /content/
!wget -q https://repo1.maven.org/maven2/org/mongodb/mongo-java-driver/3.12.10/mongo-java-driver-3.12.10.jar -P /content/

print("Dependencias instaladas.")

## 2. Importaciones y variables de entorno

Las columnas temporales se manejarán con la zona horaria de Santiago para que las consultas por fecha sean legibles.

In [ ]:
import os
import time
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from getpass import getpass

import findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"
os.environ["PYTHONPATH"] = (
    "/content/spark-3.5.7-bin-hadoop3/python/lib/py4j-0.10.9-src.zip"
)
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, round as spark_round, row_number, to_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
    IntegerType
)
from pyspark.sql.window import Window

ZONA_HORARIA = "America/Santiago"
CANTIDAD_OBJETIVO = 100
print("Variables de entorno e importaciones listas.")

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("MCDI502_Sumativa3_Grupo7")
    .config(
        "spark.jars",
        "/content/mongo-spark-connector_2.12-3.0.1.jar,"
        "/content/mongo-java-driver-3.12.10.jar"
    )
    .config("spark.sql.session.timeZone", ZONA_HORARIA)
    .getOrCreate()
)
sc = spark.sparkContext

print("Spark:", spark.version)
print("SparkSession y SparkContext creados.")

## 3. Ingesta de datos — OpenMeteo

Se consultan datos horarios de Viña del Mar para los últimos siete días. La API entrega más de 100 horas; se seleccionan **100 observaciones repartidas a lo largo de la semana**. Así, C3 trabaja sobre registros de toda la semana y el DataFrame cumple el tamaño solicitado.

Las variables corresponden a las indicadas en la pauta. El nombre técnico de cada campo se conserva para mantener el código simple.

In [ ]:
LATITUD = -33.02457
LONGITUD = -71.55183
VARIABLES_OPENMETEO = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "uv_index"
]

parametros_openmeteo = {
    "latitude": LATITUD,
    "longitude": LONGITUD,
    "hourly": ",".join(VARIABLES_OPENMETEO),
    "past_days": 7,
    "forecast_days": 1,
    "timezone": ZONA_HORARIA
}

respuesta = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params=parametros_openmeteo,
    timeout=30
)
respuesta.raise_for_status()
datos_openmeteo = respuesta.json()["hourly"]

ahora_santiago = datetime.now(ZoneInfo(ZONA_HORARIA)).replace(tzinfo=None)
inicio_semana = ahora_santiago - timedelta(days=7)

observaciones_openmeteo = []
for i, tiempo in enumerate(datos_openmeteo["time"]):
    fecha_hora = datetime.fromisoformat(tiempo)
    if inicio_semana <= fecha_hora <= ahora_santiago:
        fila = {"tiempo": fecha_hora}
        for variable in VARIABLES_OPENMETEO:
            valor = datos_openmeteo[variable][i]
            fila[variable] = float(valor) if valor is not None else None
        observaciones_openmeteo.append(fila)

if len(observaciones_openmeteo) < CANTIDAD_OBJETIVO:
    raise ValueError(
        f"OpenMeteo entregó {len(observaciones_openmeteo)} observaciones; "
        f"se requieren {CANTIDAD_OBJETIVO}."
    )

# Tomar 100 filas a intervalos regulares dentro de las últimas siete jornadas.
indices = [
    round(i * (len(observaciones_openmeteo) - 1) / (CANTIDAD_OBJETIVO - 1))
    for i in range(CANTIDAD_OBJETIVO)
]
registros_openmeteo = [observaciones_openmeteo[i] for i in indices]

esquema_openmeteo = StructType([
    StructField("tiempo", TimestampType(), False),
    StructField("temperature_2m", DoubleType(), True),
    StructField("relative_humidity_2m", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("wind_direction_10m", DoubleType(), True),
    StructField("uv_index", DoubleType(), True)
])

df_openmeteo = spark.createDataFrame(
    registros_openmeteo, schema=esquema_openmeteo
)
print("Registros OpenMeteo:", df_openmeteo.count())
df_openmeteo.show(5, truncate=False)
df_openmeteo.printSchema()

## 4. Ingesta de datos — AQICN

Se adapta el endpoint y la extracción de variables del ejemplo oficial. La pauta pide 100 registros; el código solicita 100 lecturas y conserva tanto la hora de medición informada por AQICN como la hora de consulta.

**Interpretación:** AQICN entrega el dato más reciente de una estación. Consultarlo 100 veces puede devolver la misma medición; por eso se guarda la hora de medición de la API y no se presenta cada consulta como una medición independiente.

In [ ]:
TOKEN_AQICN = "demo"  # Token de prueba utilizado en el notebook docente
CIUDAD_AQICN = "santiago"
URL_AQICN = (
    f"https://api.waqi.info/feed/{CIUDAD_AQICN}/?token={TOKEN_AQICN}"
)

registros_aqicn = []
for intento in range(CANTIDAD_OBJETIVO):
    respuesta = requests.get(URL_AQICN, timeout=30)
    respuesta.raise_for_status()
    datos = respuesta.json()

    if datos.get("status") != "ok":
        raise ValueError(f"AQICN no autorizó o no respondió correctamente: {datos}")

    informacion = datos["data"]
    iaqi = informacion.get("iaqi", {})
    tiempo_api = informacion.get("time", {}).get("s")

    registros_aqicn.append({
        "tiempo_medicion": tiempo_api,
        "hora_consulta": datetime.now(
            ZoneInfo(ZONA_HORARIA)
        ).replace(tzinfo=None),
        "PM2_5": iaqi.get("pm25", {}).get("v"),
        "PM10": iaqi.get("pm10", {}).get("v"),
        "O3": iaqi.get("o3", {}).get("v"),
        "NO2": iaqi.get("no2", {}).get("v"),
        "SO2": iaqi.get("so2", {}).get("v")
    })

    if intento < CANTIDAD_OBJETIVO - 1:
        time.sleep(1)  # Intervalo utilizado en el ejemplo del docente

esquema_aqicn = StructType([
    StructField("tiempo_medicion", StringType(), True),
    StructField("hora_consulta", TimestampType(), False),
    StructField("PM2_5", DoubleType(), True),
    StructField("PM10", DoubleType(), True),
    StructField("O3", DoubleType(), True),
    StructField("NO2", DoubleType(), True),
    StructField("SO2", DoubleType(), True)
])

# Convertir valores disponibles a número; conservar como nulo los contaminantes ausentes.
for fila in registros_aqicn:
    for variable in ["PM2_5", "PM10", "O3", "NO2", "SO2"]:
        if fila[variable] is not None:
            fila[variable] = float(fila[variable])

df_aqicn = spark.createDataFrame(registros_aqicn, schema=esquema_aqicn)
print("Registros AQICN:", df_aqicn.count())
print(
    "Horas de medición distintas informadas por AQICN:",
    df_aqicn.select("tiempo_medicion").distinct().count()
)
df_aqicn.show(5, truncate=False)
df_aqicn.printSchema()

## 5. Ingesta de datos — MongoDB (ESP32)

Se mantiene el host, la base y la colección del ejemplo docente. El usuario y la contraseña se ingresan al ejecutar, para no dejar credenciales escritas en el notebook.

El ejemplo entregado por el docente registró 28 documentos en su ejecución guardada. El notebook toma hasta 100 documentos disponibles y muestra el conteo real; si la colección aún tiene menos de 100, no inventa filas.

In [ ]:
usuario_mongo = input("Usuario de MongoDB entregado para el curso: ")
clave_mongo = getpass("Contraseña de MongoDB: ")

from urllib.parse import quote_plus
mongo_uri = (
    f"mongodb+srv://{quote_plus(usuario_mongo)}:{quote_plus(clave_mongo)}"
    "@mgtt.yr90s39.mongodb.net/?appName=MGTT"
)

base_mongo = "espdb"
coleccion_mongo = "datos"

df_mongo_origen = (
    spark.read.format("com.mongodb.spark.sql.DefaultSource")
    .option("uri", mongo_uri)
    .option("database", base_mongo)
    .option("collection", coleccion_mongo)
    .load()
)

# Nombres observados en el esquema del ejemplo docente:
# altitudFt, presionAtm, ppmCO y fecha.
df_mongo_bpm = (
    df_mongo_origen
    .select(
        col("altitudFt").alias("alturaft"),
        col("presionAtm").alias("presion_atmosferica"),
        col("ppmCO").alias("monoxido_carbono"),
        col("fecha").alias("fecha_mongo")
    )
    .orderBy(col("fecha_mongo").desc())
    .limit(CANTIDAD_OBJETIVO)
)

print("Registros MongoDB disponibles (máximo 100):", df_mongo_bpm.count())
df_mongo_bpm.show(5, truncate=False)
df_mongo_bpm.printSchema()

## 6. Ingesta de datos — MySQL mediante API REST

Se conserva la URL y el procedimiento del ejemplo docente: la base MySQL se consulta mediante el endpoint PHP que entrega JSON. Se toman los primeros 100 registros si están disponibles.

In [ ]:
URL_MYSQL = "http://mgtt.comunidadingenieria.cl/obtener_datos.php"

respuesta_mysql = requests.get(URL_MYSQL, timeout=30)
respuesta_mysql.raise_for_status()
datos_mysql = respuesta_mysql.json()

if not isinstance(datos_mysql, list):
    raise ValueError("La API de MySQL no devolvió una lista JSON.")

registros_mysql = datos_mysql[:CANTIDAD_OBJETIVO]

esquema_mysql = StructType([
    StructField("id_mysql", IntegerType(), True),
    StructField("temperatura_interior", DoubleType(), True),
    StructField("humedad", DoubleType(), True),
    StructField("fecha_mysql", StringType(), True)
])

filas_mysql = [
    (
        int(fila["id"]),
        float(fila["temperatura"]),
        float(fila["humedad"]),
        str(fila["fecha"])
    )
    for fila in registros_mysql
]

df_mysql_dht = spark.createDataFrame(filas_mysql, schema=esquema_mysql)
print("Registros MySQL disponibles (máximo 100):", df_mysql_dht.count())
df_mysql_dht.show(5, truncate=False)
df_mysql_dht.printSchema()

## 7. Control de cantidad de registros

La rúbrica pide 100 registros por fuente. Este control muestra qué fuentes alcanzan el requisito con los datos disponibles. Las filas ausentes no se generan artificialmente.

In [ ]:
conteos_fuentes = {
    "df_openmeteo": df_openmeteo.count(),
    "df_aqicn": df_aqicn.count(),
    "df_mongo_bpm": df_mongo_bpm.count(),
    "df_mysql_dht": df_mysql_dht.count()
}

for nombre, cantidad in conteos_fuentes.items():
    estado = "CUMPLE" if cantidad == CANTIDAD_OBJETIVO else "REVISAR"
    print(f"{nombre}: {cantidad} registros — {estado}")

if any(cantidad != CANTIDAD_OBJETIVO for cantidad in conteos_fuentes.values()):
    print(
        "\nAtención: solicitar al docente una fuente con 100 registros "
        "si MongoDB o MySQL aún entregan menos."
    )

## 8. Preprocesamiento y transformación

### 8.1 Conversión de altura de pies a metros

Se aplica el factor indicado en la pauta: **1 pie = 0,3048 metros**.

In [ ]:
df_mongo_bpm = df_mongo_bpm.withColumn(
    "altura_metros",
    spark_round(col("alturaft") * 0.3048, 2)
)

df_mongo_bpm.select(
    "alturaft", "altura_metros", "presion_atmosferica", "monoxido_carbono"
).show(10, truncate=False)

### 8.2 Clasificación de la condición climática

Se aplican los umbrales exactos de la rúbrica. Los valores nulos o no clasificables quedan como `CONDICION DESCONOCIDA`.

In [ ]:
df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    when(
        (col("temperature_2m") > 30) &
        (col("relative_humidity_2m") > 70),
        "CALOR Y HUMEDAD"
    ).when(
        (col("temperature_2m") > 30) &
        (col("relative_humidity_2m") <= 70),
        "CALOR SECO"
    ).when(
        col("temperature_2m") <= 30,
        "CLIMA MODERADO"
    ).otherwise("CONDICION DESCONOCIDA")
)

df_openmeteo.select(
    "temperature_2m", "relative_humidity_2m", "condicion_clima"
).show(10, truncate=False)

## 9. Integración de las cuatro fuentes

Las fuentes no comparten una clave natural. Para cumplir el ejercicio de integración se ordena cada DataFrame cronológicamente y se crea un índice de fila. Luego se unen los cuatro índices.

**Supuesto y límite:** la fila 1 de cada fuente se combina con la fila 1 de las otras, y así sucesivamente. Es una unión posicional para el ejercicio, no prueba que las cuatro mediciones ocurrieran al mismo tiempo. Se conservan las fechas de origen para que esta limitación sea visible.

In [ ]:
def agregar_indice_temporal(dataframe, columna_fecha):
    ventana = Window.orderBy(col(columna_fecha).asc())
    return dataframe.withColumn("indice", row_number().over(ventana))

openmeteo_indice = agregar_indice_temporal(df_openmeteo, "tiempo")
aqicn_indice = agregar_indice_temporal(df_aqicn, "hora_consulta")
mongo_indice = agregar_indice_temporal(df_mongo_bpm, "fecha_mongo")
mysql_indice = (
    df_mysql_dht
    .withColumn("fecha_mysql_ts", to_timestamp(col("fecha_mysql")))
)
mysql_indice = agregar_indice_temporal(mysql_indice, "fecha_mysql_ts")

df_unificado = (
    openmeteo_indice
    .join(aqicn_indice, on="indice", how="inner")
    .join(mongo_indice, on="indice", how="inner")
    .join(mysql_indice, on="indice", how="inner")
    .orderBy("indice")
)

print("Filas en df_unificado:", df_unificado.count())
df_unificado.show(10, truncate=False)
df_unificado.printSchema()

## 10. Tablas temporales para Spark SQL

Se crea una vista temporal por cada DataFrame fuente, como pide la instrucción.

In [ ]:
df_openmeteo.createOrReplaceTempView("df_openmeteo")
df_aqicn.createOrReplaceTempView("df_aqicn")
df_mongo_bpm.createOrReplaceTempView("df_mongo_bpm")
df_mysql_dht.createOrReplaceTempView("df_mysql_dht")

print("Vistas temporales creadas: df_openmeteo, df_aqicn, df_mongo_bpm, df_mysql_dht")

## 11. Consultas requeridas

### C1 — Diez temperaturas más altas de OpenMeteo

In [ ]:
consulta_c1 = spark.sql("""
    SELECT tiempo, temperature_2m
    FROM df_openmeteo
    ORDER BY temperature_2m DESC
    LIMIT 10
""")
consulta_c1.show(truncate=False)

### C2 — Humedad relativa promedio para un día y mes

Se elige el día y mes de la última observación de la muestra y se muestra explícitamente el valor consultado.

In [ ]:
fecha_objetivo = registros_openmeteo[-1]["tiempo"]
mes_dia_objetivo = fecha_objetivo.strftime("%m-%d")
print("Día y mes consultados:", fecha_objetivo.strftime("%d-%m"))

consulta_c2 = spark.sql(f"""
    SELECT
        date_format(tiempo, 'dd-MM') AS dia_mes,
        ROUND(AVG(relative_humidity_2m), 2) AS humedad_promedio
    FROM df_openmeteo
    WHERE date_format(tiempo, 'MM-dd') = '{mes_dia_objetivo}'
    GROUP BY date_format(tiempo, 'dd-MM')
""")
consulta_c2.show(truncate=False)

### C3 — Registros con precipitación superior a 2 mm durante la última semana

El DataFrame contiene 100 observaciones seleccionadas a lo largo de los últimos siete días. La consulta cuenta las observaciones de esa semana que superan 2 mm.

In [ ]:
consulta_c3 = spark.sql("""
    SELECT COUNT(*) AS registros_sobre_2_mm
    FROM df_openmeteo
    WHERE tiempo >= current_timestamp() - INTERVAL 7 DAYS
      AND precipitation > 2
""")
consulta_c3.show(truncate=False)

### C4 — Promedios de PM2.5, PM10, O3, NO2 y SO2

In [ ]:
consulta_c4 = spark.sql("""
    SELECT
        ROUND(AVG(PM2_5), 2) AS promedio_PM2_5,
        ROUND(AVG(PM10), 2) AS promedio_PM10,
        ROUND(AVG(O3), 2) AS promedio_O3,
        ROUND(AVG(NO2), 2) AS promedio_NO2,
        ROUND(AVG(SO2), 2) AS promedio_SO2
    FROM df_aqicn
""")
consulta_c4.show(truncate=False)

### C5 — Cantidad de registros con NO2 superior a 4

In [ ]:
consulta_c5 = spark.sql("""
    SELECT COUNT(*) AS registros_NO2_mayor_4
    FROM df_aqicn
    WHERE NO2 > 4
""")
consulta_c5.show(truncate=False)

### C6 — Diez registros de MongoDB, incluyendo altura en metros

In [ ]:
consulta_c6 = spark.sql("""
    SELECT
        alturaft,
        altura_metros,
        presion_atmosferica,
        monoxido_carbono
    FROM df_mongo_bpm
    LIMIT 10
""")
consulta_c6.show(truncate=False)

### C7 — Diez registros de OpenMeteo, incluyendo condición climática

In [ ]:
consulta_c7 = spark.sql("""
    SELECT
        tiempo,
        temperature_2m,
        relative_humidity_2m,
        condicion_clima
    FROM df_openmeteo
    LIMIT 10
""")
consulta_c7.show(truncate=False)

## 12. Resumen de ejecución

Este resumen permite revisar rápidamente que existan los cuatro DataFrames, la integración y las siete consultas solicitadas.

In [ ]:
print("RESUMEN DE LA SUMATIVA 3")
print("OpenMeteo:", df_openmeteo.count(), "filas")
print("AQICN:", df_aqicn.count(), "filas")
print("MongoDB:", df_mongo_bpm.count(), "filas")
print("MySQL:", df_mysql_dht.count(), "filas")
print("DataFrame integrado:", df_unificado.count(), "filas")
print("Consultas preparadas: C1, C2, C3, C4, C5, C6 y C7")

## Lista de entrega

- [ ] Ejecutar el notebook desde el inicio con **Entorno de ejecución → Ejecutar todo**.
- [ ] Revisar que cada fuente tenga 100 registros y guardar las salidas visibles.
- [ ] Confirmar que las columnas de los DataFrames y las siete consultas aparecen sin errores.
- [ ] Grabar el video de 7 a 10 minutos con participación de todos los integrantes.
- [ ] Incluir en el video las conexiones, las transformaciones, la decisión de integración y los resultados.
- [ ] Nombrar el notebook `mcdi502_s3_grupo7.ipynb` y el video `mcdi502_s3_grupo7_video.mp4`.
- [ ] Comprimir ambos archivos en un único ZIP para la entrega.

**Antes de entregar:** si MongoDB o MySQL siguen entregando menos de 100 registros, el código lo dejará visible. Se requiere que el origen disponga de 100 registros; el notebook no crea filas ficticias.